# Lab 7 V2: Sentiment Analysis  
## Data-Centric vs Model-Centric Approaches

**Student Name:** Sarah Elshiaty  
**Student ID:** S23108195  
**Course:** CS4082 / Machine Learning  

---

## Objective

The objective of this lab is to apply machine learning techniques for sentiment analysis using product review text data.

This lab compares two important approaches:

- **Model-Centric AI:** improving performance by testing different machine learning models and tuning hyperparameters.
- **Data-Centric AI:** improving performance by identifying noisy data, cleaning the dataset, and retraining the model.

In addition, a reusable model comparison function is developed to evaluate multiple classifiers using Accuracy, Precision, Recall, and F1-score.

The goal is to understand how both model selection and data quality affect classification accuracy.

# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [109]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [110]:
import pandas as pd

In [111]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
322,This is a Very good magazine and price is ex...,good
228,I've been receiving Family Handyman for a numb...,good
969,"The Mag. has a nice cover, but once inside the...",bad
639,I got bombarded with glossy junk mail and watc...,bad
421,Up to date on style and interesting articles. ...,good


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [112]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [113]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [114]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [115]:
from sklearn import metrics

In [116]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [117]:
evaluate(sgd_clf)

Accuracy: 76.6%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1 v2

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

## Task 1: Model Comparison Function

In this task, a reusable Python function is created to compare multiple machine learning models on the same dataset.

The function will:

- train each model
- make predictions on the test set
- calculate Accuracy, Precision, Recall, and F1-score
- return results in a structured DataFrame

This approach is more efficient and organized than testing models one by one.

In [118]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import LinearSVC

In [119]:
X_train = train['review']
y_train = train['label']

X_test = test['review']
y_test = test['label']

In [120]:
def compare_models(models, X_train, y_train, X_test, y_test):
    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, pos_label='good'),
            'Recall': recall_score(y_test, y_pred, pos_label='good'),
            'F1-score': f1_score(y_test, y_pred, pos_label='good')
        })

    return pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

In [121]:
models = {
    'SGDClassifier': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', SGDClassifier())
    ]),

    'MultinomialNB': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', MultinomialNB())
    ]),

    'LogisticRegression': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', LogisticRegression(max_iter=1000))
    ]),

    'RandomForest': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', RandomForestClassifier(random_state=42))
    ]),

    'LinearSVC': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', LinearSVC())
    ]),

    'AdaBoost': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', AdaBoostClassifier(random_state=42))
    ]),

    'LinearSVC Tuned': Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('clf', LinearSVC(C=0.01, max_iter=5000))
    ]),

    'MultinomialNB Tuned (alpha=5.0)': Pipeline([
        ('vect', CountVectorizer(ngram_range=(1,2))),
        ('tfidf', TfidfTransformer(use_idf=False, sublinear_tf=True)),
        ('clf', MultinomialNB(alpha=5.0))
    ]),

    'MultinomialNB Tuned (alpha=26.0)': Pipeline([
        ('vect', CountVectorizer(ngram_range=(1,2))),
        ('tfidf', TfidfTransformer(use_idf=False, sublinear_tf=True)),
        ('clf', MultinomialNB(alpha=26.0))
    ])
}

In [122]:
results_df = compare_models(models, X_train, y_train, X_test, y_test)
results_df

,Model,Accuracy,Precision,Recall,F1-score
0,MultinomialNB Tuned (alpha=26.0),0.913,0.905697,0.922,0.913776
1,MultinomialNB Tuned (alpha=5.0),0.905,0.905812,0.904,0.904905
2,LinearSVC Tuned,0.882,0.889796,0.872,0.880808
3,MultinomialNB,0.853,0.853707,0.852,0.852853
4,RandomForest,0.842,0.860759,0.816,0.837782
5,LogisticRegression,0.781,0.767619,0.806,0.786341
6,SGDClassifier,0.766,0.757752,0.782,0.769685
7,LinearSVC,0.720,0.711538,0.740,0.725490
8,AdaBoost,0.719,0.654008,0.930,0.767960


### Exercise 1 Results

The model comparison function was used to evaluate both default and tuned machine learning models on the same training and testing data.

Including the tuned models made the comparison more complete and showed how hyperparameter adjustment can improve performance.

The results showed that the tuned **Multinomial Naive Bayes** model with `alpha = 26.0` achieved the highest performance among all tested models.



## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [123]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [124]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [125]:
train.head(10)

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad
5,"The magazines are great, but I never received ...",good
6,This is one magazine I really love. It has pri...,good
7,Did not. Open.,bad
8,Forever the best magazine! Love it!!,good
9,Very disappointed. It's nothing more than an a...,bad


In [126]:
for i in range(10):
    print(train.iloc[i].to_dict())
    print('-'*80)

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}
--------------------------------------------------------------------------------
{'review': "I still have not received this.  Obviously I can't review something I haven't seen.  Where is my order???????", 'label': 'bad'}
--------------------------------------------------------------------------------
{'review': '</tr>The magazine is not worth the cost of subscription.</tr>', 'label': 'good'}
--------------------------------------------------------------------------------
{'review': 'This magazine is basically ads. Kindve worthless to me really.', 'label': 'bad'}
--------------------------------------------------------------------------------
{'review': "The only thing I've recieved, so far, is the bill.\nI know this is [ or was when I last subscribed ] a ten month 

### Exercise 2: Observations About Bad Data Points

After examining several examples from the training dataset, I noticed clear patterns in the problematic rows.

Some reviews contain HTML tags such as `<br>`, `</tr>`, and `<body>`, which suggests that the data may have been collected through web scraping and not cleaned properly.

I also observed that some reviews have incorrect labels. For example, certain reviews contain clearly negative opinions, but they are labeled as **good**. This means that some of the noisy rows may also be mislabeled.

Overall, the bad data points seem to contain HTML formatting, scraping noise, and incorrect labels. These issues can confuse the classifier and reduce prediction accuracy.

## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [127]:
def is_bad_data(review: str) -> bool:
    return '<' in review and '>' in review

### Exercise 3: Identifying Bad Data Points

To improve data quality, the placeholder function was replaced with a real rule to detect problematic reviews.

From Exercise 2, many bad rows were found to contain HTML tags such as `<br>`, `</tr>`, and `<body>`. These rows also often had noisy text or incorrect labels.

Therefore, a simple heuristic was used: if a review contains the symbols `<` and `>`, it is likely to contain HTML formatting and will be treated as bad data.

This allows the dataset to be cleaned before training the model again.

In [128]:
train_clean = train[~train['review'].map(is_bad_data)]

### Check

In [129]:
print("Original rows:", len(train))
print("Clean rows:", len(train_clean))

Original rows: 6666
Clean rows: 3999


### Cleaned Dataset Size

After applying the HTML-based filtering rule, the size of the training dataset was reduced:

- **Original rows:** 6666  
- **Clean rows:** 3999  

This shows that a large number of noisy or suspicious rows were removed. Many of these rows likely contained HTML formatting, scraping noise, and incorrect labels.

## Evaluating a model trained on the clean training set

In [130]:
from sklearn import clone

In [131]:
sgd_clf_clean = clone(sgd_clf)

In [132]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [133]:
evaluate(sgd_clf_clean)

Accuracy: 97.0%


### Model Evaluation on the Cleaned Dataset

After removing reviews that contained HTML-like formatting, the baseline SGDClassifier was trained again on the cleaned training dataset.

The cleaned model achieved an accuracy of **97.0%**, which is a major improvement compared with the original baseline accuracy of **76.6%**.

This shows that improving data quality can greatly improve model performance. In this case, cleaning the dataset produced a larger gain than simply trying different models.